In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

In [2]:
from rdkit import Chem
# Chem module is used to work with chemical structures.
# It helps convert SMILES strings into molecule objects,
# which allows us to analyze, manipulate, and extract features from molecules.

from rdkit.Chem import rdFingerprintGenerator
# rdFingerprintGenerator is used to generate molecular fingerprints.
# Fingerprints are numerical representations (binary vectors) of molecules.
# They capture structural patterns (like substructures, rings, bonds),
# which can be used as input features for machine learning models.

In [3]:
from sklearn.model_selection import *
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import *
from scipy.stats import randint

In [4]:
df=pd.read_csv("drug_discovery_final.csv")
df.head()

,canonical_smiles,molecule_chembl_id,standard_value,value,disease,target_chembl_id,standard_units,standard_type
0,N=C(N)N1CCC[C@H](NC(=O)CNC(=O)[C@@H](CCNC(=O)c...,CHEMBL117716,1270.0,1270.00,HIV,NaN,NaN,NaN
1,CCN(C(=O)C1=CCC[C@H]1C(=O)NCc1ccc(C(=N)N)cc1)C...,CHEMBL50117,890.0,0.89,HIV,NaN,NaN,NaN
2,CC(O)[C@H](NC(=O)[C@H](Cc1ccc([N+](=O)[O-])cc1...,CHEMBL108952,19.0,19.00,HIV,NaN,NaN,NaN
3,COc1cc(CNC(=O)[C@@H]2CCCN2C(=O)C(N)C(c2ccccc2)...,CHEMBL3142655,50.0,50.00,HIV,NaN,NaN,NaN
4,Cc1cc(NC(=O)Cc2ccc3[nH]c(-c4ccc(Cl)s4)nc3c2)cc...,CHEMBL337921,10000.0,10.00,HIV,NaN,NaN,NaN


In [5]:
df.shape

(19276, 8)

In [6]:
df['disease'].value_counts()

disease
Cancer          8655
HIV             5061
Dengue          4254
Tuberculosis    1306
Name: count, dtype: int64

In [7]:
df.isnull().sum()

canonical_smiles          0
molecule_chembl_id        0
standard_value            0
value                  3275
disease                   0
target_chembl_id      16001
standard_units        16004
standard_type         16001
dtype: int64

In [8]:
df=df.drop(columns=[
    "target_chembl_id",
    "standard_units",
    "standard_type",
    "value"
], errors='ignore')

In [9]:
df.columns

Index(['canonical_smiles', 'molecule_chembl_id', 'standard_value', 'disease'], dtype='str')

In [10]:
required_cols = ["disease", "molecule_chembl_id", "canonical_smiles", "standard_value"]
df = df[required_cols].copy()

In [11]:
df["standard_value"]=pd.to_numeric(df["standard_value"], errors="coerce")
df = df.dropna(subset=["standard_value"])

In [12]:
# IC50 must be positive
df = df[df["standard_value"] > 0]

In [13]:
df = df.drop_duplicates(
    subset=["molecule_chembl_id", "canonical_smiles", "disease"]
)

In [14]:
df.shape

(19146, 4)

In [15]:
df["pIC50"] = -np.log10(df["standard_value"] * 1e-9)

In [16]:
# Create a Morgan fingerprint generator
# radius=2 → considers atom neighborhoods up to 2 bonds away
# fpSize=2048 → size of fingerprint vector (2048 binary features)
morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=2048)

# Function to convert SMILES → fingerprint
def smiles_to_fp(smiles):
    # Convert SMILES string into RDKit molecule object
    mol = Chem.MolFromSmiles(smiles)
    
    # If SMILES is invalid, return None
    if mol is None:
        return None
    
    # Generate fingerprint as NumPy array (ML-friendly format)
    return morgan_gen.GetFingerprintAsNumPy(mol)

# Lists to store valid fingerprints and corresponding rows
fps = []
valid_rows = []

# Iterate through each row in dataset
for _, row in df.iterrows():
    
    # Convert SMILES to fingerprint
    fp = smiles_to_fp(row["canonical_smiles"])
    
    # Keep only valid molecules
    if fp is not None:
        fps.append(fp)           # store fingerprint
        valid_rows.append(row)   # store corresponding row

# Create new dataframe with only valid rows
df = pd.DataFrame(valid_rows).reset_index(drop=True)

# Convert list of fingerprints into NumPy array (feature matrix for ML)
X_fp = np.array(fps)

In [17]:
disease_map = {d: i for i, d in enumerate(sorted(df["disease"].unique()))}
df["disease_encoded"] = df["disease"].map(disease_map)

In [18]:
X = np.hstack([X_fp, df[["disease_encoded"]].values])
y = df["pIC50"].values

In [19]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=df["disease"]
)

In [20]:
print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)


Train shape: (15316, 2049)
Test shape: (3830, 2049)


In [21]:
rf = RandomForestRegressor(
    random_state=42,
    n_jobs=-1
)


In [22]:
param_dist = {
    "n_estimators": randint(100, 400),
    "max_depth": [None, 10, 20, 30, 40],
    "min_samples_split": randint(2, 10),
    "min_samples_leaf": randint(1, 5),
    "max_features": ["sqrt", "log2", None]
}


In [23]:
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=param_dist,
    n_iter=20,         
    scoring="r2",
    cv=3,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

In [24]:
random_search.fit(X_train, y_train)

best_rf = random_search.best_estimator_
print("\nBest Parameters:")
print(random_search.best_params_)

y_train_pred = best_rf.predict(X_train)
y_test_pred = best_rf.predict(X_test)

Fitting 3 folds for each of 20 candidates, totalling 60 fits

Best Parameters:
{'max_depth': 40, 'max_features': None, 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 108}


In [34]:
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

print("\n Train Performance")
print("Train R2   :", round(train_r2, 4))
print("Train RMSE :", round(train_rmse, 4))
print("Train MAE  :", round(train_mae, 4))

print("\n Test Performance")
print("Test R2   :", round(test_r2, 4))
print("Test RMSE :", round(test_rmse, 4))
print("Test MAE  :", round(test_mae, 4))



 Train Performance
Train R2   : 0.9559
Train RMSE : 0.3377
Train MAE  : 0.2279

 Test Performance
Test R2   : 0.7953
Test RMSE : 0.7254
Test MAE  : 0.4822


In [37]:
import numpy as np
disease_mapping = {'cancer': 0,'dengue': 1,'hiv': 2,'tuberculosis': 3}
print("\n\nEnter disease from:\n- Cancer\n- Dengue\n- HIV\n- Tuberculosis\n")
new_disease = input("Enter disease from below list: ").strip().lower()
if new_disease not in disease_mapping:
    print("Disease not recognized")
    print("Available:", list(disease_mapping.keys()))
else:
    disease_encoded = disease_mapping[new_disease]
    df_filtered = df[df['disease'].str.lower() == new_disease]
    if df_filtered.empty:
        print("No data found for this disease")
    else:
        fps = []
        valid_rows = []
        for _, row in df_filtered.iterrows():
            fp = smiles_to_fp(row["canonical_smiles"])
            if fp is not None:
                fps.append(fp)
                valid_rows.append(row)
        if len(fps) == 0:
            print("No valid molecules found")
        else:
            X_fp = np.array(fps)
            disease_col = np.full((X_fp.shape[0], 1), disease_encoded)
            X_new = np.hstack([X_fp, disease_col])
            preds = best_rf.predict(X_new)
            df_result = df_filtered.iloc[:len(preds)].copy()
            df_result["predicted_pIC50"] = preds
            df_result = df_result.sort_values(by="predicted_pIC50", ascending=False)
            print("\nTop 5 Predicted Molecules:")
            print(df_result[["canonical_smiles", "predicted_pIC50"]].head())



Enter disease from:
- Cancer
- Dengue
- HIV
- Tuberculosis



Enter disease from below list:  dengue



Top 5 Predicted Molecules:
                                        canonical_smiles  predicted_pIC50
17695  O=C1N[C@H]2[C@H](O)[C@H](O)[C@@H](O)[C@H](O)[C...         9.992003
17693  O=C1N[C@@H]2C(=C[C@H](O)[C@@H](O)[C@H]2O)c2cc3...         9.979595
17701  O=C1N[C@H]2[C@H](O)[C@H](O)[C@@H](O)C[C@H]2c2c...         9.650598
17703  O=C1N[C@H]2[C@H](O)[C@H](O)[C@@H](O)C[C@@H]2c2...         9.650598
17699  O=c1[nH]c2c(c3cc4c(c(O)c13)OCO4)C[C@H](O)[C@@H...         9.024464


The molecules displayed in the output are predicted by the model based on their pIC50 values for the selected disease. 
These molecules are not final drugs, but potential candidate compounds that may exhibit promising biological activity.
They can be used for further experimental validation, optimization, and testing as part of the drug discovery process.
Additional steps such as laboratory experiments, safety and toxicity analysis, and clinical trials are required before any molecule can be developed into an actual drug.
This system helps researchers by narrowing down the most promising candidates, thereby saving time and cost in early-stage drug discovery